# SkinDisNet: PyTorch (.pth) to Single-File ONNX Converter

This notebook converts `skindisnet_groupbalanced_best.pth` into a **single, self-contained `skindisnet_model.onnx`** (~16 MB) with all weights embedded (no external `.data` file).

In [ ]:
# Step 1: Install required libraries
!pip install -q timm onnx onnxscript onnxruntime
print("[OK] Dependencies installed successfully!")

In [ ]:
import os
import torch
import timm
import onnxruntime as ort
from google.colab import files

# Clean up any previous partial exports
for f in ["skindisnet_model.onnx", "skindisnet_model.onnx.data"]:
    if os.path.exists(f):
        os.remove(f)

# Locate checkpoint
pth_candidates = [
    "skindisnet_groupbalanced_best.pth",
    "/content/skindisnet_groupbalanced_best.pth",
    "/content/drive/MyDrive/SkinDisNet_Project/skindisnet_groupbalanced_best.pth"
]
pth_path = None
for p in pth_candidates:
    if os.path.isfile(p):
        pth_path = p
        break

if pth_path is None:
    print("Please upload 'skindisnet_groupbalanced_best.pth':")
    uploaded = files.upload()
    pth_path = list(uploaded.keys())[0]

print(f"Loading checkpoint: {pth_path}...")
ckpt = torch.load(pth_path, map_location="cpu", weights_only=False)
sd = ckpt["model_state_dict"] if "model_state_dict" in ckpt else (ckpt["state_dict"] if "state_dict" in ckpt else ckpt)
cleaned_sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}

# Instantiate architecture and load weights
model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=6)
model.load_state_dict(cleaned_sd)
model.eval()
print("[OK] Model loaded into memory successfully!")

# Export to a SINGLE self-contained ONNX file with dynamo=False
dummy_input = torch.randn(1, 3, 224, 224)
onnx_filename = "skindisnet_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    dynamo=False
)

size_mb = os.path.getsize(onnx_filename) / (1024 * 1024)
print(f"\n[SUCCESS] Exported single-file model: {onnx_filename} ({size_mb:.2f} MB)")

# Verify with ONNX Runtime right inside Colab
sess = ort.InferenceSession(onnx_filename, providers=["CPUExecutionProvider"])
print("[VERIFIED] ONNX Runtime loaded the model cleanly with zero errors!")

# Download
print("Starting download to your computer...")
files.download(onnx_filename)